# Mean-reversion strategy on the CMF backtester

This notebook shows the Group 4 Python API end to end: write a `Strategy`,
run a `Backtest` over L3 data, and inspect the results as pandas frames plus a
performance chart.

It uses a synthetic Ornstein-Uhlenbeck stream so it runs anywhere. Point
`DATA_PATH` at a folder of real `*.mbo.json` files (and set `INSTRUMENT`) to run
on the real Eurex data instead.

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
root = None
for cand in [here, *here.parents]:
    if (cand / "cmf_bt").is_dir():
        root = cand
        break
    if (cand / "python" / "cmf_bt").is_dir():
        root = cand / "python"
        break
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "examples"))

import cmf_bt  # noqa: E402
from cmf_bt import Backtest, Side, Strategy, unscale  # noqa: E402

print("cmf_bt ready, PRICE_SCALE =", cmf_bt.PRICE_SCALE)

## 1. Data
Generate a synthetic mean-reverting book, or point at the real data.

In [ ]:
from synthetic import generate

INSTRUMENT = 1000
DATA_PATH = generate("/tmp/nb_synthetic.mbo.json", instrument=INSTRUMENT, steps=3000)
print("wrote", DATA_PATH)

# Real data instead:
# DATA_PATH = '/path/to/HW_1/.../'   # folder of *.mbo.json
# INSTRUMENT = 34112

## 2. Strategy
Lean against deviations of the mid from its EWMA.

In [ ]:
class MeanReversion(Strategy):
    def __init__(self, alpha=0.02, threshold=0.0005, max_position=10):
        super().__init__()
        self.alpha, self.threshold, self.max_position = alpha, threshold, max_position
        self.ewma = None

    def on_book_update(self, ctx, u):
        if u.bid == 0 or u.ask == 0:
            return
        mid = unscale((u.bid + u.ask) // 2)
        self.ewma = (
            mid
            if self.ewma is None
            else (1 - self.alpha) * self.ewma + self.alpha * mid
        )
        dev = (mid - self.ewma) / self.ewma
        pos = ctx.position(u.instrument_id)
        if dev < -self.threshold and pos < self.max_position:
            ctx.send_limit(u.instrument_id, Side.Buy, u.ask, 1)
        elif dev > self.threshold and pos > -self.max_position:
            ctx.send_limit(u.instrument_id, Side.Sell, u.bid, 1)

## 3. Run the backtest
The `progress` callback fires on a wall-clock cadence (here 0.5s).

In [ ]:
def show_progress(info):
    print(
        f"  {info['percent'] * 100:5.1f}%  events={info['events']:>8}  pnl={info['pnl']:.4f}  {info['stats']}"
    )


bt = Backtest(pnl_sample_seconds=0.05, progress_seconds=0.5)
result = bt.run(
    MeanReversion(),
    DATA_PATH,
    instrument=INSTRUMENT,
    risk={"max_position": 10},
    progress=show_progress,
)
print("done, events:", result.events)

## 4. Results as DataFrames

In [ ]:
print("final equity:", round(result.final_equity, 4))
print("stats       :", result.stats["total"])
display(result.fills_df.head())
display(result.order_log_df["status"].value_counts())

## 5. Performance chart
Cumulative PnL with buy/sell fills (bonus deliverable).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt  # noqa: E402

result.plot()
plt.tight_layout()
plt.show()